# Pipeline de dados — trajetórias AIS

Notebook de apoio ao `src/main.py`: explica cada passo e chama as funções dos
módulos em vez de repetir código.

**Como usar:** abre a partir da raiz do repositório
(`jupyter notebook src/pipeline.ipynb`). A célula seguinte encontra a raiz e
`src/` automaticamente, com *working directory* na raiz ou em `src/`.

> Os passos 1 e 2 reescrevem os Parquet em `data/`; são reprodutíveis a partir
> do CSV original.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "data").is_dir() and (ROOT.parent / "data").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

DATA = ROOT / "data"
print("raiz:", ROOT)
print("dados:", DATA)


## 1. Limpeza do CSV

`limpar_csv_para_parcelar` lê só as colunas úteis, ordena por `(id, t)`,
converte o rótulo para binário (`fishing` → 1) e grava `ais_123.parquet`.


In [ ]:
from limpar_csv import limpar_csv_para_parcelar

ais_path = limpar_csv_para_parcelar(
    DATA / "123_fishing_trajs.csv",
    DATA / "ais_123.parquet",
)
ais_path


## 2. Divisão, `t_rel` e normalização

A divisão é feita por **trajetória completa** (70/15/15, semente 42), para não
haver fuga de informação. O `t_rel` fica em bruto e a normalização (mediana e
IQR) é ajustada **só ao treino**; os parâmetros ficam em `normalizacao.json`.


In [ ]:
from preparar_dados import preparar_dados

treino_path, teste_path, validacao_path, params_path = preparar_dados(
    DATA / "ais_123.parquet",
    DATA / "treino_norm.parquet",
    DATA / "teste_norm.parquet",
    DATA / "validacao_norm.parquet",
    70, 15, 15,
    params_path=DATA / "normalizacao.json",
)
treino_path, teste_path, validacao_path, params_path


### Verificação da divisão

`verificar_divisao` confirma que nenhuma trajetória foi partida e que os pontos
de pesca se mantêm.


In [ ]:
from verificar_divisao import verificar_divisao

verificacao = verificar_divisao(
    DATA / "ais_123.parquet",
    treino_path, teste_path, validacao_path,
)
verificacao["ok"]


### Distribuição das features normalizadas

Mediana e IQR foram ajustados só ao treino; as estatísticas dos três conjuntos
devem ficar próximas de 0 e numa escala comparável.


In [ ]:
import pandas as pd

treino = pd.read_parquet(treino_path)
features = ["x", "y", "signed_turn", "time_gap", "euc_speed", "distanceToShore"]
treino[features].describe(percentiles=[.01, .05, .5, .95, .99]).T[
    ["min", "1%", "50%", "99%", "max"]
]


### Porque cortar em gaps > 10 min?

Gaps longos podem ser falhas de transmissão. Se uma janela os atravessasse,
juntaria pontos que não são contíguos no tempo. O maior gap do dataset é uma
falha de transmissão (13,1 h com apenas 1,55 km), não duas viagens fundidas.


In [ ]:
from analisar_gaps import analisar_gaps, resumo_gaps

gaps = analisar_gaps(ais_path)
resumo_gaps(gaps).head(5)


## 3. Janelamento com sobreposição

Cada trajetória é cortada em janelas de **30 min** com **50% de sobreposição**.
Parâmetros decididos:

- `gap_max_s=600`: nenhuma janela atravessa gaps > 10 min;
- `min_pontos=30`: evita janelas quase vazias;
- `T` não é fixo: o maior nº de pontos define `T` (590 no treino) e as restantes
  janelas levam padding (`mascara=False`, `y=-1`).

O modo alternativo `n_pontos=256` fica para comparação.


In [ ]:
from criar_janelas import criar_janelas

r = criar_janelas(
    treino_path,
    duracao_s=30 * 60,
    sobreposicao_pct=50,
    gap_max_s=600,
    min_pontos=30,
)
r["info"]


### O que observar

- `X`: `(nº de janelas, T, nº de features)`.
- `mascara`: `True` nos pontos reais; o resto é padding.
- `meta`: `id`, posições e `t_rel` de cada janela (para reconstruir segmentos).


In [ ]:
n_pontos_treino = len(treino)
sem = r["info"]["pontos_descartados"]
print("X:", r["X"].shape)
print(f"pontos sem janela: {sem:,} de {n_pontos_treino:,} "
      f"({100 * sem / n_pontos_treino:.2f}%)")
print(f"padding: {100 * (1 - r['mascara'].mean()):.1f}% de X")
r["meta"]["n_pontos"].describe(percentiles=[.05, .5, .95])


### Comparação com o modo pontos

Mesma sobreposição e mesmo corte em gaps, mas janelas de 256 pontos:


In [ ]:
r_pontos = criar_janelas(
    treino_path, n_pontos=256, sobreposicao_pct=50, gap_max_s=600,
)
r_pontos["info"]


## 4. Guardar as janelas

`criar_janelas` aceita `output_path` e grava um `.npz` comprimido, que pode ser
relido com `carregar_janelas` (sem repetir o processamento). Exemplo:

```python
criar_janelas(
    treino_path, duracao_s=30 * 60, sobreposicao_pct=50,
    gap_max_s=600, min_pontos=30,
    output_path=DATA / "janelas_treino.npz",
)
```


## Próximos passos

- gerar e guardar as janelas de treino, validação e teste (3.3);
- decidir a política para os pontos sem janela (3.4);
- implementar o *batching* com máscara/packing (3.5);
- correr as ablações de sobreposição, duração e features (3.6).

Plano completo em
[`docs/markdown/passos_futuros.md`](../docs/markdown/passos_futuros.md).
